In [ ]:
# =========================
# DATASET DOWNLOAD
# =========================

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip plantvillage-dataset.zip

!unzip segmented_sam3_10.zip

from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d nirmalsankalana/plantdoc-dataset
!unzip plantdoc-dataset.zip

!pip install timm

In [4]:
# =========================
# IMPORTS
# =========================

import torch
import torch.nn as nn
import timm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset
from torch.utils.data import WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import os
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# CLASS MAPPING
# =========================

class_mapping = {

    "Corn_Gray_leaf_spot": 0,
    "Corn_leaf_blight": 1,
    "Corn_rust_leaf": 2,
    "Tomato_Septoria_leaf_spot": 3,
    "Tomato_leaf": 4,
    "Apple_Scab_Leaf": 5,
    "Apple_leaf": 6,
    "Apple_rust_leaf": 7,
    "grape_leaf": 8,
    "grape_leaf_black_rot": 9,

    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": 0,
    "Corn_(maize)___Northern_Leaf_Blight": 1,
    "Corn_(maize)___Common_rust_": 2,
    "Tomato___Septoria_leaf_spot": 3,
    "Tomato___healthy": 4,
    "Apple___Apple_scab": 5,
    "Apple___healthy": 6,
    "Apple___Cedar_apple_rust": 7,
    "Grape___healthy": 8,
    "Grape___Black_rot": 9,
}


# =========================
# DATA AUGMENTATION
# =========================

train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(224,scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])


# =========================
# PLANTDOC DATASET
# =========================

class PlantDocDataset(Dataset):

    def __init__(self,root,transform=None):

        self.samples=[]
        self.transform=transform

        for class_name in os.listdir(root):

            if class_name in class_mapping:

                class_path=os.path.join(root,class_name)

                for img in os.listdir(class_path):

                    self.samples.append(
                        (os.path.join(class_path,img),
                         class_mapping[class_name])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):

        img_path,label=self.samples[idx]

        image=Image.open(img_path).convert("RGB")

        if self.transform:
            image=self.transform(image)

        return image,label


# =========================
# LOAD PLANTDOC TRAIN DATA
# =========================

plantdoc_root="segmented_sam3_new_5"

full_pd_dataset=PlantDocDataset(
    plantdoc_root,
    transform=train_transform
)

indices=list(range(len(full_pd_dataset)))

train_idx,val_idx=train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

pd_train_subset=Subset(full_pd_dataset,train_idx)

pd_val_dataset=PlantDocDataset(
    plantdoc_root,
    transform=val_transform
)

pd_val_subset=Subset(pd_val_dataset,val_idx)

print("PlantDoc train size:",len(pd_train_subset))
print("PlantDoc val size:",len(pd_val_subset))


# =========================
# PLANTVILLAGE DATASET
# =========================

class PlantVillageDataset(Dataset):

    def __init__(self,root,transform=None):

        self.samples=[]
        self.transform=transform

        for class_name in os.listdir(root):

            if class_name in class_mapping:

                class_path=os.path.join(root,class_name)

                for img in os.listdir(class_path):

                    if img.lower().endswith((".jpg",".jpeg",".png")):

                        self.samples.append(
                            (os.path.join(class_path,img),
                             class_mapping[class_name])
                        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):

        img_path,label=self.samples[idx]

        image=Image.open(img_path).convert("RGB")

        if self.transform:
            image=self.transform(image)

        return image,label


plantvillage_root="plantvillage/segmented"

pv_dataset=PlantVillageDataset(
    plantvillage_root,
    transform=train_transform
)

print("PlantVillage size:",len(pv_dataset))


# =========================
# COMBINE DATASETS
# =========================

joint_train_dataset=ConcatDataset([
    pv_dataset,
    pd_train_subset
])


# =========================
# WEIGHTED SAMPLER
# =========================

targets=[]

for dataset in joint_train_dataset.datasets:
    for _,label in dataset:
        targets.append(label)

targets=np.array(targets)

class_count=np.bincount(targets)

class_weights=1.0/class_count

sample_weights=class_weights[targets]

sampler=WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)


# =========================
# DATALOADERS
# =========================

train_loader=DataLoader(
    joint_train_dataset,
    batch_size=2,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader=DataLoader(
    pd_val_subset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# =========================
# LOAD DINOv2 LARGE
# =========================

model=timm.create_model(
    "vit_large_patch14_dinov2.lvd142m",
    pretrained=True,
    img_size=224
)

for param in model.parameters():
    param.requires_grad=False

for param in model.blocks[-3:].parameters():
    param.requires_grad=True

in_features=model.num_features

model.head=nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features,10)
)

model=model.to(device)


# =========================
# LOSS + OPTIMIZER
# =========================

criterion=nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer=torch.optim.AdamW(
    filter(lambda p:p.requires_grad,model.parameters()),
    lr=1e-5,
    weight_decay=1e-4
)

scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)


# =========================
# TRAINING LOOP
# =========================

from torch.amp import GradScaler,autocast

def train_model(model,train_loader,val_loader,epochs=30,patience=8):

    scaler=GradScaler("cuda")

    best_acc=0
    early_stop=0

    for epoch in range(epochs):

        model.train()

        running_loss=0

        for images,labels in train_loader:

            images=images.to(device)
            labels=labels.to(device)

            optimizer.zero_grad()

            with autocast("cuda"):
                outputs=model(images)
                loss=criterion(outputs,labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss+=loss.item()

        model.eval()

        correct=0
        total=0

        with torch.no_grad():

            for images,labels in val_loader:

                images=images.to(device)
                labels=labels.to(device)

                with autocast("cuda"):
                    outputs=model(images)

                _,preds=torch.max(outputs,1)

                total+=labels.size(0)
                correct+=(preds==labels).sum().item()

        val_acc=correct/total

        print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f} | Val Acc={val_acc:.4f}")

        if val_acc>best_acc:
            best_acc=val_acc
            torch.save(model.state_dict(),"best_dino_large.pth")
            early_stop=0
        else:
            early_stop+=1

        if early_stop>=patience:
            print("Early stopping triggered.")
            break

        scheduler.step()

    print("Best Validation Accuracy:",best_acc)


train_model(model,train_loader,val_loader,epochs=30)

Using device: cuda
PlantDoc train size: 746
PlantDoc val size: 187
PlantVillage size: 10205
Epoch 1: Loss=0.4491 | Val Acc=0.7754
Epoch 2: Loss=0.3653 | Val Acc=0.8128
Epoch 3: Loss=0.3381 | Val Acc=0.8235
Epoch 4: Loss=0.3261 | Val Acc=0.8396
Epoch 5: Loss=0.3197 | Val Acc=0.8075
Epoch 6: Loss=0.3123 | Val Acc=0.8235
Epoch 7: Loss=0.3071 | Val Acc=0.8396
Epoch 8: Loss=0.3048 | Val Acc=0.8610
Epoch 9: Loss=0.3023 | Val Acc=0.8503
Epoch 10: Loss=0.2962 | Val Acc=0.8396
Epoch 11: Loss=0.2969 | Val Acc=0.8610
Epoch 12: Loss=0.2965 | Val Acc=0.8503
Epoch 13: Loss=0.2939 | Val Acc=0.8396
Epoch 14: Loss=0.2927 | Val Acc=0.8717
Epoch 15: Loss=0.2897 | Val Acc=0.8824
Epoch 16: Loss=0.2869 | Val Acc=0.8663
Epoch 17: Loss=0.2872 | Val Acc=0.8449
Epoch 18: Loss=0.2873 | Val Acc=0.8503
Epoch 19: Loss=0.2862 | Val Acc=0.8770
Epoch 20: Loss=0.2876 | Val Acc=0.8717
Epoch 21: Loss=0.2859 | Val Acc=0.8663
Epoch 22: Loss=0.2849 | Val Acc=0.8610
Epoch 23: Loss=0.2867 | Val Acc=0.8663
Early stopping trigg

In [5]:

# =========================
# LOAD TEST DATA
# =========================

!unzip plantdoc_test2.zip

test_transform=val_transform

plantdoc_test_root="plantdoc_test2"

pd_test_dataset=PlantDocDataset(
    plantdoc_test_root,
    transform=test_transform
)

test_loader=DataLoader(
    pd_test_dataset,
    batch_size=2,
    shuffle=False
)

print("PlantDoc Test size:",len(pd_test_dataset))


# =========================
# LOAD BEST MODEL
# =========================

model=timm.create_model(
    "vit_large_patch14_dinov2.lvd142m",
    pretrained=False,
    img_size=224
)

model.head=nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.num_features,10)
)

model.load_state_dict(
    torch.load("best_dino_large.pth",map_location=device)
)

model=model.to(device)
model.eval()

print("Best DINO model loaded successfully.")


# =========================
# TEST EVALUATION
# =========================

correct=0
total=0
all_preds=[]
all_labels=[]

with torch.no_grad():

    for images,labels in test_loader:

        images=images.to(device)
        labels=labels.to(device)

        outputs=model(images)

        _,preds=torch.max(outputs,1)

        total+=labels.size(0)
        correct+=(preds==labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc=correct/total

print("\n==============================")
print("DINOv2 LARGE Test Accuracy:",test_acc)
print("==============================")

cm=confusion_matrix(all_labels,all_preds)

print("\nConfusion Matrix:")
print(cm)


class_names=[
"Corn_Gray_leaf_spot",
"Corn_leaf_blight",
"Corn_rust_leaf",
"Tomato_Septoria_leaf_spot",
"Tomato_leaf",
"Apple_Scab_Leaf",
"Apple_leaf",
"Apple_rust_leaf",
"grape_leaf",
"grape_leaf_black_rot"
]

print("\nPer Class Accuracy:")

for i,class_name in enumerate(class_names):

    class_total=cm[i].sum()
    class_correct=cm[i][i]

    acc=class_correct/class_total if class_total>0 else 0

    print(f"{class_name}: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels,all_preds,target_names=class_names))

Archive:  plantdoc_test2.zip
replace plantdoc_test2/Apple_leaf/download (3).jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: plantdoc_test2/Apple_leaf/download (3).jpg  
  inflating: plantdoc_test2/Apple_leaf/images (1).jpg  
  inflating: plantdoc_test2/Apple_leaf/images (2).jpg  
  inflating: plantdoc_test2/Apple_leaf/images (3).jpg  
  inflating: plantdoc_test2/Apple_leaf/images.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_1.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_2.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_3.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_4.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_5.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_6.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_7.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_8.jpg  
  inflating: plantdoc_test2/Apple_leaf/test_Apple leaf_9.jpg  
  inflating: plantdoc_test2/Apple_